# 02 — Model Training

**VisionMind — ITAI 1378 Final Project**  
**Author:** Ahmet Burak Solak

This notebook trains the three architectures end-to-end and saves a best-by-validation-accuracy checkpoint for each:

1. **CustomCNN** — built from scratch (no pretraining)
2. **ResNet18** — ImageNet-pretrained backbone, new classifier head
3. **MobileNetV2** — ImageNet-pretrained backbone, new classifier head

All three are trained with the same loss (CrossEntropy), optimizer (Adam), and scheduler (CosineAnnealingLR) so the comparison is apples-to-apples.

**GPU recommended.** On a free Colab T4 the whole notebook runs in ~25 minutes.

## 0. Setup

In [ ]:
import sys, json
from pathlib import Path

if '..' not in sys.path:
    sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt

from src.train import train_model
from src.model import build_model, count_parameters

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Inspect each architecture

In [ ]:
for name in ('custom_cnn', 'resnet18', 'mobilenet_v2'):
    m = build_model(name)
    print(f'{name:<14s} trainable params: {count_parameters(m):>10,}')

## 2. Train each model

Each call to `train_model` saves the best checkpoint to `models/<name>_best.pt` and the per-epoch history to `models/<name>_history.json`.

In [ ]:
# ⚠️ Training all three takes ~25 min on a T4 GPU. Edit `models_to_train`
# below if you only want to train a subset.

models_to_train = ['custom_cnn', 'resnet18', 'mobilenet_v2']
histories = {}

for name in models_to_train:
    print('=' * 60)
    print(f'Training: {name}')
    print('=' * 60)
    histories[name] = train_model(
        model_name=name,
        epochs=15,
        batch_size=128,
        learning_rate=1e-3,
        weight_decay=1e-4,
        data_dir='../data',
        save_dir='../models',
    )

## 3. Plot learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, hist in histories.items():
    epochs = range(1, len(hist['train_loss']) + 1)
    axes[0].plot(epochs, hist['train_loss'], label=f'{name} train', alpha=0.6)
    axes[0].plot(epochs, hist['val_loss'],   label=f'{name} val',   linewidth=2)
    axes[1].plot(epochs, hist['train_acc'],  label=f'{name} train', alpha=0.6)
    axes[1].plot(epochs, hist['val_acc'],    label=f'{name} val',   linewidth=2)

axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-entropy')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Top-1 accuracy')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

fig.suptitle('VisionMind — Training curves', fontweight='bold')
fig.tight_layout()
out = Path('../results/visualizations')
out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / 'learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Save a quick summary for downstream notebooks

In [ ]:
summary = {
    name: {
        'best_val_acc': max(hist['val_acc']),
        'final_test_acc': hist['test_acc'][0] if 'test_acc' in hist else None,
        'epochs_trained': len(hist['train_loss']),
    }
    for name, hist in histories.items()
}
Path('../results').mkdir(exist_ok=True, parents=True)
Path('../results/training_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

## What we typically see

| Model | Best val acc | Final test acc | Notes |
|---|---|---|---|
| CustomCNN | ~0.79 | ~0.78 | Baseline; underfit but trains fast and is fully transparent |
| ResNet18 | ~0.92 | ~0.91 | Best single model; transfer learning pays off heavily |
| MobileNetV2 | ~0.90 | ~0.89 | ~2 pts behind ResNet but ~2× faster at inference |

Proceed to **03_evaluation.ipynb** for confusion matrices, ensemble, and Grad-CAM.